In [1]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from datetime import datetime
from random import random

2023-04-16 13:35:33.627597: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-04-16 13:35:33.713760: I tensorflow/core/util/port.cc:104] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2023-04-16 13:35:33.716186: W tensorflow/compiler/xla/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2023-04-16 13:35:33.716198: I tensorflow/compiler/xla/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudar

In [2]:
#Simulate Time Series on 3000 days
time_ = np.arange(0, 3000, 1)/10
labels = 1 + np.sin(time_) + time_*0.01

# Simulate 10% of missing labels. Mark them with -1
ind_nan=np.random.choice(len(labels), int(len(labels)*0.1), replace=False)
labels[ind_nan]=-1

#Create sample_weight, an array of 1's with 0 on the missing labels
sample_weight=np.ones(len(labels))
sample_weight[ind_nan]=0
#Create dataframes with data and weights
df = pd.DataFrame({'labels': labels})
sample_weight_df = pd.DataFrame(dict(sample_weight=sample_weight))

In [3]:
sample_weight_df

,sample_weight
0,1.0
1,1.0
2,1.0
3,1.0
4,1.0
...,...
2995,0.0
2996,1.0
2997,1.0
2998,0.0


In [4]:
x = np.array([1,2,3])
x[0] = np.array([4])

In [5]:
x

array([4, 2, 3])

In [4]:
data = [['2021-08-24 09:00:00', 1, 0.4, 200, -1]
        , ['2021-08-24 09:00:00', 2, 0.3, 10, 5]
        , ['2021-08-24 09:00:00', 3, 0.4, 240, 0.6]
        ,['2021-08-25 09:00:00', 1, 1.6, 200, -1]
        , ['2021-08-25 09:00:00', 2, 0, 10, 5]
        , ['2021-08-25 09:00:00', 3, 0.5, 240, 0.6]
        ,['2021-08-26 09:00:00', 1, 2.6, 200, -1]
        , ['2021-08-26 09:00:00', 2, 0.1, 10, 5]
        , ['2021-08-26 09:00:00', 3, 2.5, 240, 0.6]
        ,['2021-08-27 09:00:00', 1, 0.6, 200, -1]
        , ['2021-08-27 09:00:00', 2, 4.1, 10, 5]
        , ['2021-08-27 09:00:00', 3, 0.5, 240, 0.6]
       ]
  
# Create the pandas DataFrame
df = pd.DataFrame(data, columns=['tempo','coord', 'chuva','altura','declividade'])
  
# print dataframe.
df

,tempo,coord,chuva,altura,declividade
0,2021-08-24 09:00:00,1,0.4,200,-1.0
1,2021-08-24 09:00:00,2,0.3,10,5.0
2,2021-08-24 09:00:00,3,0.4,240,0.6
3,2021-08-25 09:00:00,1,1.6,200,-1.0
4,2021-08-25 09:00:00,2,0.0,10,5.0
5,2021-08-25 09:00:00,3,0.5,240,0.6
6,2021-08-26 09:00:00,1,2.6,200,-1.0
7,2021-08-26 09:00:00,2,0.1,10,5.0
8,2021-08-26 09:00:00,3,2.5,240,0.6
9,2021-08-27 09:00:00,1,0.6,200,-1.0


In [12]:
df = df.sort_values(['tempo','coord'])

In [13]:
df

,tempo,coord,chuva,altura,declividade
0,2021-08-24 09:00:00,1,0.4,200,-1.0
1,2021-08-24 09:00:00,2,0.3,10,5.0
2,2021-08-24 09:00:00,3,0.4,240,0.6
3,2021-08-25 09:00:00,1,1.6,200,-1.0
4,2021-08-25 09:00:00,2,0.0,10,5.0
5,2021-08-25 09:00:00,3,0.5,240,0.6
6,2021-08-26 09:00:00,1,2.6,200,-1.0
7,2021-08-26 09:00:00,2,0.1,10,5.0
8,2021-08-26 09:00:00,3,2.5,240,0.6
9,2021-08-27 09:00:00,1,0.6,200,-1.0


In [ ]:
qtd_tempo = 4
qtd_coord = 3

In [16]:
X_train = df.to_numpy().reshape(qtd_tempo,qtd_coord,-1)

In [17]:
X_train

array([[['2021-08-24 09:00:00', 1, 0.4, 200, -1.0],
        ['2021-08-24 09:00:00', 2, 0.3, 10, 5.0],
        ['2021-08-24 09:00:00', 3, 0.4, 240, 0.6]],

       [['2021-08-25 09:00:00', 1, 1.6, 200, -1.0],
        ['2021-08-25 09:00:00', 2, 0.0, 10, 5.0],
        ['2021-08-25 09:00:00', 3, 0.5, 240, 0.6]],

       [['2021-08-26 09:00:00', 1, 2.6, 200, -1.0],
        ['2021-08-26 09:00:00', 2, 0.1, 10, 5.0],
        ['2021-08-26 09:00:00', 3, 2.5, 240, 0.6]],

       [['2021-08-27 09:00:00', 1, 0.6, 200, -1.0],
        ['2021-08-27 09:00:00', 2, 4.1, 10, 5.0],
        ['2021-08-27 09:00:00', 3, 0.5, 240, 0.6]]], dtype=object)

time_step = X_train.shape[1]
input_dim = X_train.shape[2] #qtd colunas 
out = Y_train.shape[2]

In [11]:
X_train.shape

(3, 4, 5)

In [1]:
import numpy as np

a = np.arange(6).reshape((3, 2))
print(a)



[[0 1]
 [2 3]
 [4 5]]


In [2]:
b = a.reshape((2, 3))
print(b)



[[0 1 2]
 [3 4 5]]


In [3]:
c = b.reshape((1, -1))
print(c)

[[0 1 2 3 4 5]]


In [3]:
s = pd.Series([0, 1, np.nan, 3])
s



0    0.0
1    1.0
2    NaN
3    3.0
dtype: float64

In [4]:
s.interpolate()


0    0.0
1    1.0
2    2.0
3    3.0
dtype: float64

In [5]:
s

0    0.0
1    1.0
2    NaN
3    3.0
dtype: float64

In [2]:
df = pd.read_csv('df_limpo.csv')

In [3]:
df

,Unnamed: 0,n_coord,id_estacao,ocorrencia,lon_ocr,lat_ocr,id_tempo_x,tempo,vintequatrohoras,Altitude_numerica,...,flg_comunidades,flg_agricola,flg_exploracao_mineral,flg_rocha,flg_cobertura_vegetal,flg_afloramento_rochoso,flg_favela,flg_ocupacao_desordenada,flg_areas_de_risco,graurisc
0,0,3,509,0,-43.067260,-22.984866,12259261,2020-04-23 09:00:00.000,0.0,52.628016,...,0,0,0,0,1,0,0,0,0,0
1,1,4,509,0,-43.065324,-22.984232,12259261,2020-04-23 09:00:00.000,0.0,55.637476,...,0,0,0,0,1,0,0,0,0,0
2,2,5,509,0,-43.066912,-22.983404,12259261,2020-04-23 09:00:00.000,0.0,49.241599,...,0,0,0,0,1,0,0,0,0,0
3,3,6,509,0,-43.065536,-22.982592,12259261,2020-04-23 09:00:00.000,0.0,25.035910,...,0,0,0,0,1,0,0,0,0,0
4,4,7,509,0,-43.050843,-22.980390,12259261,2020-04-23 09:00:00.000,0.0,58.199686,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2195007,3269238,3422,527,0,-43.088680,-22.866286,11930941,2019-09-08 09:00:00.000,0.0,41.686992,...,1,0,0,0,0,0,1,0,0,0
2195008,3269239,3455,527,0,-43.087010,-22.863764,11930941,2019-09-08 09:00:00.000,0.0,22.700953,...,0,0,0,0,0,0,0,0,0,0
2195009,3269240,3466,527,0,-43.088237,-22.865197,11930941,2019-09-08 09:00:00.000,0.0,41.149896,...,1,0,0,0,0,0,1,0,0,0
2195010,3269241,3489,527,0,-43.087003,-22.862870,11930941,2019-09-08 09:00:00.000,0.0,25.621156,...,0,0,0,0,0,0,0,0,0,0
